[Reference](https://medium.com/@inprogrammer/how-to-turn-your-laptop-into-a-private-ai-server-in-30-minutes-no-gpu-required-4adf37c38525$0)

# Installing llama.cpp
## macOS (the fastest path)
```
brew install llama.cpp
```

## Linux (build from source)
```
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
cmake -B build
cmake --build build --config Release -j$(nproc)
```

```
cmake -B build -DGGML_CUDA=ON
cmake --build build --config Release -j$(nproc)
```

## Windows

```
📁 local-ai/
├── 📁 models/
│ ├── 🟦 Qwen3–8B-Q4_K_M.gguf
│ └── 🟦 Mistral-7B-Q4_K_M.gguf
├── 📄 server.py
├── 📄 batch_process.py
└── 📄 requirements.txt
```

# Getting Your First Model


## Download a Model
```
llama-cli --hf-repo bartowski/Qwen3-8B-GGUF \
          --hf-file Qwen3-8B-Q4_K_M.gguf \
          -p "Tell me about Python type hints" \
          -n 256
```

```
# Install huggingface-hub first
pip install huggingface-hub

# Then pull the model
huggingface-cli download bartowski/Qwen3-8B-GGUF \
  Qwen3-8B-Q4_K_M.gguf \
  --local-dir ./models
```

# Running Your First Chat
```
llama-cli \
  -m ./models/Qwen3-8B-Q4_K_M.gguf \
  --chat-template qwen3 \
  -cnv \
  -n 512
```

```
llama-server \
  -m ./models/Qwen3-8B-Q4_K_M.gguf \
  --host 0.0.0.0 \
  --port 8080 \
  -n 2048 \
  --ctx-size 8192
```

In [ ]:
from openai import OpenAI

# Point to localhost instead of api.openai.com
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="not-needed"  # llama-server doesn't require auth by default
)
response = client.chat.completions.create(
    model="local-model",  # any string works
    messages=[
        {"role": "system", "content": "You are a helpful Python expert."},
        {"role": "user", "content": "Explain Python's GIL in simple terms."}
    ],
    temperature=0.7,
    max_tokens=500,
)
print(response.choices[0].message.content)

stream = client.chat.completions.create(
    model="local-model",
    messages=[{"role": "user", "content": "Write a Python decorator that logs function calls."}],
    stream=True
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

# Batch Processing Without Fear


In [ ]:
import os
from pathlib import Path
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="local")
def summarize_file(filepath: str) -> str:
    content = Path(filepath).read_text()
    response = client.chat.completions.create(
        model="local-model",
        messages=[
            {
                "role": "system",
                "content": "Summarize the following code file in 3 bullet points."
            },
            {
                "role": "user",
                "content": content[:4000]  # respect context window
            }
        ],
        max_tokens=300
    )
    return response.choices[0].message.content
# Process every .py file in a directory
python_files = list(Path("./src").glob("**/*.py"))
for file in python_files:
    print(f"\n--- {file.name} ---")
    summary = summarize_file(str(file))
    print(summary)